# Analise de Seguranca Cibernetica e Ataques Digitais

Este notebook documenta a analise exploratoria da base `simulacao_ciberseguranca_brasil.csv`, conforme as instrucoes do Projeto G2 - Tema 26.

## Contextualizacao

A transformacao digital ampliou a superficie de ataque de organizacoes publicas e privadas. A analise de incidentes ajuda a identificar ameacas recorrentes, vulnerabilidades criticas, setores mais afetados e periodos de maior risco.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

DATA_PATH = Path('..') / 'dados' / 'simulacao_ciberseguranca_brasil.csv'
df = pd.read_csv(DATA_PATH)
df['data'] = pd.to_datetime(df['data'])
df.head()

## Explicacao da base

A base possui dados simulados de incidentes ciberneticos no Brasil, incluindo ano, mes, data, regiao, UF, setor, tipo de ataque, vulnerabilidade, incidentes, impacto financeiro, tempo de recuperacao, sistemas afetados, criticidade e status de resposta.

In [ ]:
df.info()
df.isna().sum()

## Limpeza e preparacao

A base nao apresenta valores ausentes. A coluna `data` foi convertida para datetime e foram criados campos auxiliares para analise temporal.

In [ ]:
df['periodo'] = df['data'].dt.to_period('M').astype(str)
df['trimestre'] = df['data'].dt.quarter
df.describe(include='all')

## KPIs

In [ ]:
kpis = {
    'total_incidentes': int(df['incidentes'].sum()),
    'tipo_ataque_predominante': df.groupby('tipo_ataque')['incidentes'].sum().idxmax(),
    'setor_mais_afetado': df.groupby('setor')['incidentes'].sum().idxmax(),
    'impacto_financeiro_total': float(df['impacto_financeiro'].sum()),
    'tempo_medio_recuperacao': float(df['tempo_recuperacao'].mean()),
    'regiao_mais_critica': df.groupby('regiao')['incidentes'].sum().idxmax(),
}
kpis

## Visualizacoes

In [ ]:
temporal = df.groupby('periodo', as_index=False)['incidentes'].sum()
px.line(temporal, x='periodo', y='incidentes', title='Evolucao temporal dos incidentes')

In [ ]:
ataques = df.groupby('tipo_ataque', as_index=False)['incidentes'].sum().sort_values('incidentes')
px.bar(ataques, x='incidentes', y='tipo_ataque', orientation='h', title='Incidentes por tipo de ataque')

In [ ]:
setores = df.groupby('setor', as_index=False)['incidentes'].sum().sort_values('incidentes')
px.bar(setores, x='incidentes', y='setor', orientation='h', title='Incidentes por setor')

In [ ]:
heatmap = df.pivot_table(values='incidentes', index='ano', columns='mes', aggfunc='sum', fill_value=0)
px.imshow(heatmap, aspect='auto', title='Heatmap mensal de incidentes')

In [ ]:
scatter = df.groupby(['ano', 'mes', 'regiao', 'setor', 'tipo_ataque'], as_index=False).agg(
    incidentes=('incidentes', 'sum'),
    impacto_financeiro=('impacto_financeiro', 'sum'),
    tempo_recuperacao=('tempo_recuperacao', 'mean'),
)
px.scatter(scatter, x='incidentes', y='impacto_financeiro', color='tipo_ataque', size='tempo_recuperacao', title='Impacto financeiro x incidentes')

## Interpretacao

A leitura dos KPIs e visualizacoes permite identificar concentracoes por periodo, setor, regiao e tipo de ataque. O dashboard Streamlit complementa esta analise com filtros interativos para explorar cenarios especificos.

## Conclusao

O projeto evidencia como uma base estruturada de incidentes pode apoiar decisoes de seguranca cibernetica, priorizando vulnerabilidades recorrentes, periodos criticos e areas com maior impacto operacional e financeiro.